In [ ]:
import torch
import ultralytics

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    
print(f"Ultralytics Version: {ultralytics.__version__}")

/home/nick/Documents/CPE542/Super-Fast-and-Optimized-Fire-Detection-Embedded-System-Project/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


ModuleNotFoundError: No module named 'ultralytics'

**Import Data and Train Model**

In [ ]:
import kagglehub
from ultralytics import YOLO
import os
import yaml

In [ ]:
# download dataset
ds_path = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
print(f"Dataset downloaded to: {ds_path}")

# patch YAML for local use
orig_yaml = os.path.join(ds_path, "data.yaml")
with open(orig_yaml, "r") as f:
    data = yaml.safe_load(f)

# update the root path to the local kagglehub directory
data["path"] = ds_path

local_yaml = "local_smoke_fire.yaml"
with open(local_yaml, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

# train locally
model = YOLO("yolo26n.pt") 
model.train(
    data=local_yaml, 
    epochs=50, 
    imgsz=640, 
    batch=16,     
    device=0, 
    workers=4,    
    amp=True      
)
# export to tensorflow
model.export(format="saved_model")
model.export(format="onnx")

**Resume Model if Training Interrupted**

In [ ]:
model = YOLO("runs/detect/train6/weights/last.pt") 
model.train(resume=True)
model.export(format="saved_model")
model.export(format="onnx")

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

def predict_and_show(model_path, image_path, conf=0.25, imgsz=640):
    """
    run YOLO inference on an image and display bounding boxes
    """
    # load model
    model = YOLO(model_path)

    # run prediction
    results = model.predict(
        source=image_path,
        conf=conf,
        imgsz=imgsz,
        save=False,
        verbose=False
    )

    # ultralytics gives us an annotated image already
    annotated_img = results[0].plot()

    # convert BGR to RGB for matplotlib
    annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

    # show image
    plt.figure(figsize=(8, 8))
    plt.imshow(annotated_img)
    plt.axis("off")
    plt.title("YOLO Predictions")
    plt.show()


predict_and_show(
    model_path="runs/detect/train6/weights/best.pt",
    image_path="data/train/images/WEB09307.jpg",
    conf=0.2
)
